In [ ]:
import sys 
sys.path.append("C:/Users/jonge094/PycharmProjects/ms2query_2_0/ms_chemical_space_explorer")



In [16]:
from matchms.importing import load_from_mgf
from tqdm import tqdm

neg_val_spectra = list(tqdm(load_from_mgf("../../../ms2deepscore/data/pytorch/new_corinna_included/training_and_validation_split/negative_validation_spectra.mgf")))
neg_test_spectra = list(tqdm(load_from_mgf("../../../ms2deepscore/data/pytorch/new_corinna_included/training_and_validation_split/negative_testing_spectra.mgf")))


7551it [00:05, 1452.14it/s]
7142it [00:04, 1502.96it/s]


In [17]:
from ms2deepscore.models import load_model
from ms2deepscore.models import compute_embedding_array

ms2deepscore_model = load_model("../../../ms2deepscore/data/pytorch/new_corinna_included/trained_models/both_mode_precursor_mz_ionmode_10000_layers_500_embedding_2024_11_21_11_23_17/ms2deepscore_model.pt")

embeddings = compute_embedding_array(ms2deepscore_model, neg_val_spectra)

C:\Users\jonge094\AppData\Local\miniconda3\envs\ms2query2\lib\site-packages\ms2deepscore\models\load_model.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_setting

In [124]:
import numpy as np
more_test_embeddings = np.tile(embeddings, (70, 1))


In [125]:
more_test_embeddings.shape

(528570, 500)

In [127]:
import pynndescent
import time
start_time = time.time()
ann_model = pynndescent.NNDescent(more_test_embeddings, metric="cosine", n_neighbors=30)
ann_model.prepare()
print("Time eleapsed: " + str(time.time() - start_time))

Time eleapsed: 163.06616854667664


In [128]:
start_time = time.time()
ann_model.update(embeddings[:2])
ann_model.prepare()

print("Time eleapsed: " + str(time.time() - start_time))

Time eleapsed: 86.74623227119446


In [129]:
start_time = time.time()
indices, dists = ann_model.query(embeddings[:1000], epsilon=1, k=1000)
print("Time eleapsed: " + str(time.time() - start_time))

Time eleapsed: 1.8495116233825684


In [103]:
for dist in dists:
    if dist > 0.000001:
        print(dist)

In [88]:
correct = 0
not_correct = 0
for i, index in enumerate(indices):
    correct_if_0 = index[0]%7551-i
    if correct_if_0== 0:
        correct += 1
    else:
        not_correct +=1
print(correct)
print(not_correct)

6783
768


In [117]:
more_test_embeddings.shape

(226530, 500)

In [126]:
from ms2deepscore.vector_operations import cosine_similarity_matrix
start_time = time.time()
matrix = cosine_similarity_matrix(more_test_embeddings, embeddings[:1000])
print("Time eleapsed: " + str(time.time() - start_time))

Time eleapsed: 14.737722158432007


In [120]:
matrix

array([[1.        , 0.88234181, 0.73878687, ..., 0.55531438, 0.58179732,
        0.61927229],
       [0.88234181, 1.        , 0.91034399, ..., 0.50462923, 0.51674736,
        0.54464448],
       [0.73878687, 0.91034399, 1.        , ..., 0.48358345, 0.4835752 ,
        0.54332801],
       ...,
       [0.49057829, 0.55037322, 0.56818589, ..., 0.39834881, 0.41004598,
        0.5298201 ],
       [0.48655507, 0.57605234, 0.5946298 , ..., 0.40762756, 0.42247458,
        0.50066275],
       [0.41914688, 0.47922786, 0.48648942, ..., 0.38071478, 0.41207848,
        0.47297686]])